In [20]:
# Please execute/shift-return this cell everytime you run the notebook.  Don't edit it. 
%load_ext autoreload
%autoreload 2
from notebook import * 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Matrix tiling algorithm

Let's try to partition GEMM into smaller tiles!

In [2]:
render_code("matrix_mul/blockmm.c", show=["//START","//END"])

// matrix_mul/blockmm.c:59-77 (19 lines)
//START
void blockmm(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  uint64_t i,j,k, ii, jj, kk;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
              for(kk = k; kk < k+tile_size; kk++)
                c[ii][jj] += a[ii][kk]*b[kk][jj];
      }
    }
  }  
}
//END

In [3]:
! cd matrix_mul/; make clean blockmm

rm -f blockmm mm blockmm_transpose cachegrind.* mm_dump rect_blockmm_trans blockmm_transpose_reg blockmm_reg
gcc -O4 -DHAVE_LINUX_PERF_EVENT_H blockmm.c perfstats.c -o blockmm 


## Try with tile size == 32

In [4]:
! cd matrix_mul; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm.csv
for size in (512, 1024, 2048):
    ! ./matrix_mul/blockmm {size} 32 >> ./matrix_mul/blockmm.csv

In [5]:
#display_df_mono(render_csv("matrix_mul/mm.csv"))
#display_df_mono(render_csv("matrix_mul/blockmm.csv"))
df_baseline = render_csv("matrix_mul/mm.csv")
df_baseline.insert(df_baseline.columns.get_loc('index'), 'configuration', 'baseline')
df_block = render_csv("matrix_mul/blockmm.csv")
df_block.insert(df_block.columns.get_loc('index'), 'configuration', 'tiling')
merged_df = pd.concat([df_baseline, df_block], ignore_index=True)
display_df_mono(merged_df)

,configuration,index,size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses,tile_size
0,baseline,0,512,1070666322,764811009,0.714332,0.182178,0.139332,0.244983,130916838,534392529,nan
1,baseline,1,1024,8622850290,11448002071,1.327635,0.179112,2.050477,0.233301,1004366110,4305014345,nan
2,baseline,2,2048,69010943458,126378488103,1.831282,0.179144,22.639970,0.309702,10672034660,34459077979,nan
3,baseline,3,4096,552923697065,1468398611124,2.655698,0.179135,263.041099,0.518777,143189253195,276013239373,nan
4,tiling,0,512,1093652553,384247813,0.351344,0.189630,0.072865,0.214568,115516152,538366510,32.000000
5,tiling,1,1024,8942224249,3181356173,0.355768,0.179291,0.570390,0.210076,924656621,4401543172,32.000000
6,tiling,2,2048,71537519616,27947769438,0.390673,0.179120,5.006012,0.217487,7658013962,35211312547,32.000000


## Try with tile size == 8

In [21]:
for size in (512, 1024, 2048):
    ! ./matrix_mul/blockmm {size} 8 >> ./matrix_mul/blockmm.csv 

In [22]:
#display_df_mono(render_csv("matrix_mul/mm.csv"))
#display_df_mono(render_csv("matrix_mul/blockmm.csv"))
df_baseline = render_csv("matrix_mul/mm.csv")
df_baseline.insert(df_baseline.columns.get_loc('index'), 'configuration', 'baseline')
df_block = render_csv("matrix_mul/blockmm.csv")
df_block.insert(df_block.columns.get_loc('index'), 'configuration', 'tiling')
merged_df = pd.concat([df_baseline, df_block], ignore_index=True)
display_df_mono(merged_df)

,configuration,index,size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses,tile_size
0,baseline,0,512,1055345676,725437770,0.687394,0.186749,0.135475,0.240146,126484891,526699450,nan
1,baseline,1,1024,8613829844,11433457382,1.327337,0.179260,2.049566,0.232245,998788845,4300582600,nan
2,baseline,2,2048,69007256610,123208944010,1.785449,0.179111,22.068129,0.310622,10703302500,34457667827,nan
3,baseline,3,4096,553031805372,1530510999378,2.767492,0.179115,274.137870,0.519088,143295918429,276053384241,nan
4,tiling,0,512,996940815,349877362,0.350951,0.221157,0.077378,0.214524,105277392,490749735,32.000000
5,tiling,1,1024,8942231731,3180329766,0.355653,0.179018,0.569335,0.209540,922297195,4401526619,32.000000
6,tiling,2,2048,71543445707,28001752125,0.391395,0.179037,5.013361,0.217664,7664850039,35214159070,32.000000
7,tiling,3,512,1266367168,261413991,0.206428,0.218160,0.057030,0.008980,5353196,596130338,8.000000
8,tiling,4,1024,10130837054,2228894193,0.220011,0.217549,0.484894,0.009366,44667454,4768970764,8.000000
9,tiling,5,2048,81054275338,20851069736,0.257248,0.218093,4.547463,0.010176,388251769,38154480218,8.000000


In [23]:
! cd matrix_mul; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm.csv
for tile_size in (4, 8, 16, 32, 64, 128, 256, 512):
    ! ./matrix_mul/blockmm 2048 {tile_size} >> ./matrix_mul/blockmm.csv
display_df_mono(render_csv("matrix_mul/blockmm.csv"))

,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,4,97989031491,24820691423,0.253301,0.180202,4.472726,0.014968,654663147,43736739436
1,1,2048,8,81046421111,21678740379,0.267485,0.179087,3.882370,0.010471,399498162,38151636842
2,2,2048,16,74454895637,19319055376,0.259473,0.179218,3.462315,0.071887,2594839551,36096130281
3,3,2048,32,71528375968,27889766028,0.389912,0.179206,4.998015,0.217313,7650861312,35206681872
4,4,2048,64,70150966595,32665777663,0.465650,0.179080,5.849790,0.242466,8436477084,34794520178
5,5,2048,128,69471570156,36634734483,0.527334,0.179124,6.562154,0.244795,8467659184,34590750507
6,6,2048,256,69148907402,43878814661,0.634555,0.179139,7.860407,0.245414,8465527781,34494856443
7,7,2048,512,69002926023,62533162869,0.906239,0.179244,11.208690,0.237515,8182596065,34450859290


In [9]:
render_code("matrix_mul/blockmm_reg.c", show=["//START","//END"])

// matrix_mul/blockmm_reg.c:59-82 (24 lines)
//START
void blockmm(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  uint64_t i,j,k, ii, jj, kk;
    double result = 0;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
                {
                result = 0;
                for(kk = k; kk < k+tile_size; kk++)
                    result += a[ii][kk]*b[kk][jj];
                c[ii][jj] += result;
          }
      }
    }
  }  
}
//END

In [10]:
! cd matrix_mul; make blockmm_reg; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_reg.csv
for tile_size in (4, 8, 16, 32, 64, 128, 256, 512):
    ! ./matrix_mul/blockmm_reg 2048 {tile_size} >> ./matrix_mul/blockmm_reg.csv
display_df_mono(render_csv("matrix_mul/blockmm_reg.csv"))

gcc -O4 -DHAVE_LINUX_PERF_EVENT_H blockmm_reg.c perfstats.c -o blockmm_reg 
blockmm_reg.c: In function ‘main’:
blockmm_reg.c:48:16: warning: format ‘%lu’ expects argument of type ‘long unsigned int’, but argument 3 has type ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wformat=-Wformat=]8;;]
   48 |   printf("%d,%lu,",ARRAY_SIZE,tile_size);
      |              ~~^              ~~~~~~~~~
      |                |              |
      |                |              int
      |                long unsigned int
      |              %u


,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,4,106229999405,27619481009,0.259997,0.179102,4.946715,0.017846,752532949,42168824376
1,1,2048,8,78811742665,19609444084,0.248814,0.179054,3.511156,0.013409,412805496,30785413010
2,2,2048,16,66921533551,15586990700,0.232914,0.179061,2.791023,0.100450,2605394204,25937163335
3,3,2048,32,61314817568,18373993000,0.299666,0.179289,3.294260,0.296211,7012102783,23672694872
4,4,2048,64,58579843189,20743873879,0.354113,0.179100,3.715237,0.378661,8547627693,22573317862
5,5,2048,128,57224253827,20336169181,0.355377,0.179064,3.641477,0.393907,8677643915,22029695526
6,6,2048,256,56548711458,20103212469,0.355503,0.179245,3.603394,0.398147,8663383507,21759265146
7,7,2048,512,56262874669,44051569311,0.782960,0.179196,7.893857,0.399509,8646699885,21643311536
